# Chapter 16 — Generalization: Splits, Validation, and Leakage

*From Absolute Zero* — companion notebook.

Every block below is the code printed in the chapter, in the same order. Run the cells top to bottom; the output should match the book exactly. If it does not, check `requirements.txt` first, then the errata page.

In [ ]:
!pip -q install -r https://raw.githubusercontent.com/USER/from-absolute-zero/main/requirements.txt  # Colab only; skip locally

## Create the data

Run once. Every dataset in this book is generated by code you can read — nothing is downloaded, so nothing can rot behind a dead link.

In [ ]:
import numpy as np, pandas as pd
rng = np.random.default_rng(11)
n = 1470
tenure   = np.clip(rng.gamma(2.2, 3.0, n), 0.2, 40).round(1)
salary   = np.clip(rng.normal(65000, 18000, n), 25000, 160000).round(-2)
overtime = (rng.random(n) < 0.28).astype(int)
commute  = np.clip(rng.gamma(2.0, 6.0, n), 1, 60).round(0)
satis    = np.clip(rng.normal(3.3, 0.95, n), 1, 5).round(1)
promo    = np.clip(rng.gamma(1.6, 1.6, n), 0, 15).round(1)
dept     = rng.choice(["Sales", "R&D", "Support"], n, p=[0.32, 0.45, 0.23])
z = (-2.55 + 1.15*overtime - 0.135*tenure - 0.60*(satis - 3.3)
     + 0.024*commute + 0.095*promo - 0.000014*(salary - 65000)
     + np.where(dept == "Sales", 0.45,
                np.where(dept == "Support", 0.20, 0.0)))
left = (rng.random(n) < 1 / (1 + np.exp(-z))).astype(int)
hr = pd.DataFrame({"Department": dept, "YearsAtCompany": tenure,
    "MonthlyIncome": (salary/12).round(0),
    "OverTime": np.where(overtime == 1, "Yes", "No"),
    "CommuteMinutes": commute, "JobSatisfaction": satis,
    "YearsSincePromotion": promo, "Attrition": left})
hr.to_csv("hr.csv", index=False)
print(f"{len(hr):,} employees, attrition rate {hr['Attrition'].mean():.1%}")

## Shared setup

Imports and the objects the blocks below reuse. The chapter prints these once and then continues the same session.

In [ ]:
import pandas as pd, numpy as np, warnings; warnings.filterwarnings("ignore")
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import (train_test_split, cross_val_score,
                                     StratifiedKFold, GroupKFold)
from sklearn.metrics import roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
hr = pd.read_csv("hr.csv")
X = pd.get_dummies(hr.drop(columns="Attrition"),
                   columns=["Department", "OverTime"],
                   drop_first=True).astype(float)
y = hr["Attrition"].values
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3,
                                      random_state=7, stratify=y)
cv = StratifiedKFold(5, shuffle=True, random_state=0)

## The chapter code

### Block 1  (`c1.py`)

In [ ]:
hr = pd.read_csv("hr.csv")
X = pd.get_dummies(hr.drop(columns="Attrition"),
                   columns=["Department", "OverTime"],
                   drop_first=True).astype(float)
y = hr["Attrition"].values

Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3,
                                      random_state=7, stratify=y)
print(f"train {len(Xtr):,}  test {len(Xte):,}")
print(f"base rate: train {ytr.mean():.4f}  test {yte.mean():.4f}")
print("the test set is now closed until the final step")

### Block 2  (`c2.py`)

In [ ]:
print(f"{'depth':>6}{'train AUC':>11}{'test AUC':>10}{'leaves':>8}")
for d in [1, 2, 3, 5, 8, 12, None]:
    t = DecisionTreeClassifier(max_depth=d, random_state=0)
    t.fit(Xtr, ytr)
    a_tr = roc_auc_score(ytr, t.predict_proba(Xtr)[:, 1])
    a_te = roc_auc_score(yte, t.predict_proba(Xte)[:, 1])
    print(f"{str(d):>6}{a_tr:>11.4f}{a_te:>10.4f}{t.get_n_leaves():>8}")

### Block 3  (`c3.py`)

In [ ]:
for d in [2, 3, 4, 5, 6, 8, 12]:
    s = cross_val_score(DecisionTreeClassifier(max_depth=d, random_state=0),
                        Xtr, ytr, cv=cv, scoring="roc_auc")
    print(f"depth {d:>2}: AUC {s.mean():.4f} +/- {s.std():.4f}"
          f"   folds {np.round(s, 3)}")

### Block 4  (`c4.py`)

In [ ]:
pipe = Pipeline([("scale", StandardScaler()),
                 ("model", LogisticRegression(max_iter=1000))])
best_depth = 4
cands = {"tree (depth 4)": DecisionTreeClassifier(max_depth=best_depth,
                                                 random_state=0),
         "logistic":       pipe}
for name, m in cands.items():
    s = cross_val_score(m, Xtr, ytr, cv=cv, scoring="roc_auc")
    print(f"{name:<16} CV AUC {s.mean():.4f} +/- {s.std():.4f}")

### Block 5  (`c5.py`)

In [ ]:
pipe = Pipeline([("scale", StandardScaler()),
                 ("model", LogisticRegression(max_iter=1000))])
final = pipe.fit(Xtr, ytr)
p_test = final.predict_proba(Xte)[:, 1]

cv_est = cross_val_score(pipe, Xtr, ytr, cv=cv, scoring="roc_auc").mean()
tree = DecisionTreeClassifier(max_depth=4, random_state=0).fit(Xtr, ytr)
print(f"cross-validated estimate: {cv_est:.4f}")
print(f"test set result:          {roc_auc_score(yte, p_test):.4f}")
print(f"(the tree, for reference: "
      f"{roc_auc_score(yte, tree.predict_proba(Xte)[:, 1]):.4f})")

### Block 6  (`c6.py`)

In [ ]:
pipe = Pipeline([("scale", StandardScaler()),
                 ("model", LogisticRegression(max_iter=1000))])
p_test = pipe.fit(Xtr, ytr).predict_proba(Xte)[:, 1]

budget = 60
order = np.argsort(p_test)[::-1][:budget]
caught = yte[order].sum()
expected = budget * yte.mean()
print(f"AUC {roc_auc_score(yte, p_test):.3f} on {len(yte)} held-out employees")
print(f"calling the top {budget}: finds {caught} of {yte.sum()} leavers")
print(f"random {budget}: would find {expected:.1f}")
print(f"lift: {caught/expected:.1f}x")

### Block 7  (`c_leak.py`)

In [ ]:
from sklearn.datasets import load_breast_cancer
Xc, yc = load_breast_cancer(return_X_y=True)
k = StratifiedKFold(5, shuffle=True, random_state=0)

# WRONG: the scaler sees every row, including each fold's validation rows
leaked = StandardScaler().fit_transform(Xc)
a = cross_val_score(LogisticRegression(max_iter=5000), leaked, yc,
                    cv=k, scoring="roc_auc").mean()

# RIGHT: the pipeline refits the scaler inside every fold
clean = cross_val_score(
    Pipeline([("s", StandardScaler()),
              ("m", LogisticRegression(max_iter=5000))]),
    Xc, yc, cv=k, scoring="roc_auc").mean()

print(f"leaked {a:.4f}   clean {clean:.4f}   difference {a - clean:+.4f}")

### Block 8  (`c_group.py`)

In [ ]:
# 600 tickets triaged by four agents. Each agent has their own habit: some
# escalate far more than others. A model given anything that identifies the
# agent can predict the agent's habit instead of doing the task.
rng = np.random.default_rng(0)
n, agent = 600, None
agent = rng.integers(0, 4, n)                    # the repeated entity
habit = np.array([0.10, 0.40, 0.60, 0.90])[agent]  # base escalation rate
signal = rng.normal(0, 1, n)                     # the real, weak evidence

Xg = np.c_[signal * 0.6,                          # weak genuine signal
           agent, habit + rng.normal(0, .02, n)]  # agent fingerprint
yg = (rng.random(n) < np.clip(habit + 0.10 * signal, 0, 1)).astype(int)

m = LogisticRegression(max_iter=1000)
rand = cross_val_score(m, Xg, yg, scoring="roc_auc",
                       cv=StratifiedKFold(4, shuffle=True,
                                          random_state=0)).mean()
grp = cross_val_score(m, Xg, yg, groups=agent, scoring="roc_auc",
                      cv=GroupKFold(4)).mean()
print(f"random 4-fold   AUC {rand:.4f}   (agents appear on both sides)")
print(f"grouped 4-fold  AUC {grp:.4f}   (each agent held out entirely)")
print(f"the random split flatters the model by {rand - grp:+.4f}")